# Русский wake-word «Помогатор»

Этот notebook заменяет английский генератор официального quick-start четырьмя русскими Piper-голосами. Результат — `pomogator.onnx`; Vosk после пробуждения продолжит распознавать команду. Данные имеют смешанные лицензии: модель предназначена только для личного некоммерческого POC.

In [ ]:
# 1. Закреплённые исходники и зависимости
import os, sys, locale
locale.getpreferredencoding = lambda do_setlocale=True: 'UTF-8'
!git clone --depth 1 --branch feature/vosk-local-wakeword https://github.com/IavnFGV/ph-server.git
!git clone https://github.com/dscripka/openWakeWord.git openwakeword
!git -C openwakeword checkout 368c03716d1e92591906a84949bc477f3a834455
!git clone https://github.com/dscripka/piper-sample-generator.git piper-sample-generator-legacy
!git -C piper-sample-generator-legacy checkout f1988a4d54eddb23d99e86f0adfef6226a85acc7
!pip install -q -e ./openwakeword --no-deps
!pip install -q git+https://github.com/rhasspy/piper-sample-generator.git@2971426a55072f7d22fec416ca7800df8bd23207
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 onnxruntime==1.22.1 onnx==1.19.1 onnxsim pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19
os.makedirs('openwakeword/openwakeword/resources/models', exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O openwakeword/openwakeword/resources/models/melspectrogram.onnx
print('Project commit:')
!git -C ph-server rev-parse HEAD
print('CUDA:', __import__('torch').cuda.is_available())

In [ ]:
# 2. Четыре русских голоса и синтетические positive/hard-negative WAV
from pathlib import Path
voices = Path('voices'); voices.mkdir(exist_ok=True)
base = 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ru/ru_RU'
for name in ('denis', 'dmitri', 'irina', 'ruslan'):
    stem = f'ru_RU-{name}-medium.onnx'
    url = f'{base}/{name}/medium/{stem}'
    if not (voices / stem).exists():
        !wget -q --show-progress -O {voices / stem} {url}
    if not (voices / (stem + '.json')).exists():
        !wget -q -O {voices / (stem + '.json')} {url + '.json'}
!python ph-server/training/pomogator/generate_russian_samples.py --voices-dir voices --output-dir my_custom_model/pomogator --train-samples 1000 --test-samples 500 --overwrite
from IPython.display import Audio, display
example = next(Path('my_custom_model/pomogator/positive_test').glob('*.wav'))
print('Контрольный пример:', example)
display(Audio(str(example)))

In [ ]:
# 3. Данные официального notebook: RIR, фон, музыка и отрицательные признаки
import numpy as np, scipy, datasets
from tqdm import tqdm
Path('mit_rirs').mkdir(exist_ok=True)
if not Path('MIT_environmental_impulse_responses').exists():
    !git lfs install
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
rir_files = list(Path('MIT_environmental_impulse_responses/16khz').glob('*.wav'))
for source in tqdm(rir_files, desc='RIR'):
    target = Path('mit_rirs') / source.name
    if not target.exists():
        rate, audio = scipy.io.wavfile.read(source)
        scipy.io.wavfile.write(target, rate, audio)
Path('audioset').mkdir(exist_ok=True)
if not Path('audioset/bal_train09.tar').exists():
    !wget -q --show-progress -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
if not Path('audioset/audio').exists():
    !tar -xf audioset/bal_train09.tar -C audioset
Path('audioset_16k').mkdir(exist_ok=True)
audio_files = list(Path('audioset/audio').glob('**/*.flac'))
audio_ds = datasets.Dataset.from_dict({'audio': [str(p) for p in audio_files]}).cast_column('audio', datasets.Audio(sampling_rate=16000))
for row in tqdm(audio_ds, desc='AudioSet'):
    out = Path('audioset_16k') / (Path(row['audio']['path']).stem + '.wav')
    if not out.exists(): scipy.io.wavfile.write(out, 16000, (row['audio']['array'] * 32767).astype(np.int16))
Path('fma').mkdir(exist_ok=True)
if not any(Path('fma').glob('*.wav')):
    fma = iter(datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True).cast_column('audio', datasets.Audio(sampling_rate=16000)))
    for _ in tqdm(range(120), desc='FMA (1 hour)'):
        row = next(fma); out = Path('fma') / (Path(row['audio']['path']).stem + '.wav')
        scipy.io.wavfile.write(out, 16000, (row['audio']['array'] * 32767).astype(np.int16))
if not Path('openwakeword_features_ACAV100M_2000_hrs_16bit.npy').exists():
    !wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
if not Path('validation_set_features.npy').exists():
    !wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print('Данные готовы')

In [ ]:
# 4. Аугментация, обучение и выгрузка ONNX
import shutil, hashlib, datetime
shutil.copy('ph-server/training/pomogator/pomogator.yml', 'pomogator.yml')
!python openwakeword/openwakeword/train.py --training_config pomogator.yml --augment_clips
!python openwakeword/openwakeword/train.py --training_config pomogator.yml --train_model
model = Path('my_custom_model/pomogator.onnx')
assert model.exists(), 'ONNX model was not created'
sha = hashlib.sha256(model.read_bytes()).hexdigest().upper()
report = Path('training-report.txt')
report.write_text(f'model=pomogator.onnx\nsha256={sha}\ndate_utc={datetime.datetime.now(datetime.timezone.utc).isoformat()}\n', encoding='utf-8')
print(report.read_text())
from google.colab import files
files.download(str(model))
files.download(str(report))